# Full Bridge Rectifier with Capacitor Filter - Analysis

![Schematic of full bridge rectifier circuit with capacitor filter](https://github.com/ravichandrakorivi/power-electronic-design/raw/main/images/rectifier_cap_filter_schema.png)

![Plot of full bridge rectifier circuit with capacitor filter](https://github.com/ravichandrakorivi/power-electronic-design/raw/main/images/rectifier_cap_filter_plot.png)

**Spice Netlist**:
```net
* D:\Academics\spice-circuit-simulation\ltspice-tutorials\power-electronic-design\rectifier_cap_filter.asc
* Generated by LTspice 24.1.10 for Windows.
D1 c o D
D2 0 c D
D3 b o D
D4 0 b D
Cf o 0 1000µ
Ro o 0 25
Rs c a 0.1
Vs a b SINE(0 325 50)
.model D D
.lib C:\Users\adeet\AppData\Local\LTspice\lib\cmp\standard.dio
.tran 100m
.backanno
.end
```

# Full Bridge Rectifier with Capacitor Filter - Design

![Design of full bridge rectifier circuit with capacitor filter](https://github.com/ravichandrakorivi/power-electronic-design/raw/main/images/rectifier_cap_filter_design.jpg)

## Specifications

- $V_o$
- Ripple, $\Delta V_r$
- Output power, $P_o$
- Input voltage, $V_m = \sqrt{2} V_{rms}$
- Input voltage tolerance =  10 %
- Frequency, $f_s =  50 \text{ Hz}$

## Design of Capacitance, $C$

$$ \frac{1}{2}CV_{m1}^2 - \frac{1}{2}CV_{m2}^2 = \left(\frac{\pi - \alpha}{\pi}\right) P_o \frac{T}{2} $$

$$ \frac{C}{2}(V_{m1}^2 - V_{m2}^2) = \left(\frac{\pi - \alpha}{\pi}\right) \frac{P_o}{2f}  $$

$$ C\left(\frac{V_{m1} + V_{m2}}{2}\right) (V_{m1} - V_{m2}) = \left(\frac{\pi - \alpha}{\pi}\right) \frac{P_o}{2f} $$

$$ C V_o \cdot \Delta V_r = \left(\frac{\pi - \alpha}{\pi}\right) \frac{P_o}{2f} $$

$$ C =  \left(\frac{\pi - \alpha}{\pi}\right) \frac{P_o}{2fV_o \cdot \Delta V_r}$$

$$ C = \left(\frac{\pi - \alpha}{\pi}\right) \frac{I_o}{2f \cdot \Delta V_r} $$ 

Therefore, the value of $C$ should be designed for the maximum value of $I_o$. The value of $C$ must be sufficient to handle the maximum $I_o$ and minimum value of $\Delta V_r$.

## Conduction Angle, $\alpha$

$$ V_{m2} = V_{m1} \cos \alpha $$
$$ \alpha = \cos^{-1} \left( \frac{V_{m2}}{V_{m1}} \right )$$ 

Where, 
- $ V_{m1} = \sqrt{2} V_{rms} $
- $ V_{m2} = V_{m1} - \Delta{V_r} $

## Finding maximum possible $I_o$

RMS current through the capacitor:

$$ I_{c, rms} = \sqrt{(I_m - I_o)^2 \cdot \frac{\alpha}{\pi} + I_o^2 \cdot \left(\frac{\pi - \alpha}{\pi}\right)} $$

## Capacitor Selection

$$ C = \left(\frac{\pi - \alpha}{\pi}\right) \frac{I_{o, max}}{2f \cdot \Delta V_{r, min}} $$

$$ \text{Voltage rating } = \sqrt{2} V_{rms} \left(1 + \frac{\text{\%tol}}{100}\right) $$ 

$$I_{c,rms} = \sqrt{(I_m - I_o)^2 \cdot \frac{\alpha}{\pi} + I_o^2 \cdot \left(\frac{\pi - \alpha}{\pi}\right)} $$
$$ \text{Type: Aluminum Electrolyte} $$

## Diode Selection

$$ PIV = \sqrt{2} \cdot V_{rms} \cdot \left(1 + \frac{\text{\%tol}}{100}\right) $$ 

$$ I_{d, av} = I_m \left(\frac{\alpha}{2\pi}\right) $$ 

$$ I_{d, m} = I_m $$ 

## Calculations

In [1]:
import math

# Specifications of the circuit
vi_rms = 230    # rms value of the input AC voltage in volts
tol    = 20     # percentage tolerance of the input voltage
vr     = 30     # peak-to-peak ripple voltage at the output in volts
f      = 50     # mains frequency in Hz
Po     = 100    # output power in watts


# Calucations for rectifier filter module
v1min = math.sqrt(2) * vi_rms * (1 - (tol/100))
v1max = math.sqrt(2) * vi_rms * (1 + (tol/100))

v2min = v1min - vr
v2max = v1max - vr

a_max = math.acos(v2min/v1min) # maximum diode conduction angle
a_min = math.acos(v2max/v1max) # minimum diode conduction angle

# Capacitor Selection
C = ((math.pi - a_max)/math.pi) * (Po/(f*(v1min**2 + v2min**2)))*1e6 # capacitance value in uF

Vomin = (v1min + v2min) / 2           # minimum dc value of output voltage
Vonom = math.sqrt(2)*vi_rms - (vr/2)  # nominal dc value of output voltage
Vomax = (v1max + v2max) / 2           # maximum dc value of output voltage
Iomax = Po / Vomin                    # maximum average load current
Im = Iomax * math.pi / a_min          # peak diode current
Icrms = math.sqrt((Im-Iomax)**2*(a_min/math.pi) + Iomax**2*((math.pi-a_min)/math.pi)) # rms current in cap

# Diode selection
piv = v1max     # peak inverse voltage
Idavg = Im*a_max/(2*math.pi) # avg current throught the diode
Idrms = Im*math.sqrt(a_max/(2*math.pi)) # rms current though diode

In [2]:
print(f"****Capacitor selection****")
print(f"Capacitor value : {C} uF")
print(f"Voltage rating > {Vomax} V")
print(f"RMS rating > {Icrms} A")

print("\n")

print(f"****Diode Selection****")
print(f"piv : {piv} V")
print(f"Average current rating : {Idavg} A")
print(f"RMS current rating : {Idrms} A")

****Capacitor selection****
Capacitor value : 14.01104527431035 uF
Voltage rating > 375.32294321497426 V
RMS rating > 1.0759362357481685 A


****Diode Selection****
piv : 390.32294321497426 V
Average current rating : 0.2505589675974594 A
RMS current rating : 0.9019106174918028 A
